<font color='red' size='6'>REMINDER</font>

# Fill in any place that says `YOUR CODE HERE`.
- You should remove the line that says `raise NotImplementedError()`. If you do not, your code will (unsurprisingly) throw a run-time error and cause everything to fail.
- Do **NOT** write your answer anywhere else other than where it says `YOUR CODE HERE`. Simply write your code directly below this comment in the **same code cell**.

# Make sure everything runs as expected.
- Go to the menubar, select *Kernel* > *Restart & Run All*

# Do <ins>NOT</ins> change the title (i.e., file name) of this notebook.

# Do <ins>NOT</ins> delete any of the cells in this notebook.

# Make sure you save your work
- Go to the menubar, select *File* > *Save and Checkpoint*

# Continuing with the Pallet Movers Case

We continue work with our client, Pallet Movers. 

## Run the Next 2 Code Cells

Run the next two code cells before attempting to complete the tasks. After running both code cells, you will have the following variables in memory and available for use in your tasks below.

- A `list` variable named `hubs` that contains the cities where the hubs are located.
- A `list` variable named `spokes` that contains the cities where the spokes are located.
- A `list` variable named `requests` that contain the requests from potential customers for Pallet Movers to move their freight.

After running the code cell, you should examine these lists to understand their structure, content, etc.

In [1]:
from nose.tools import assert_equal, assert_in, assert_is_instance
from nose.tools import assert_almost_equal, assert_not_equal

In [2]:
import pandas as pd
hubs = ['Pittsburgh', 'Newark', 'Richmond', 'Charlotte', 'Atlanta']
spokes = ['Akron', 'Youngstown', 'Allentown', 'Harrisburg', 'Morgantown',
          'White Plains', 'Trenton', 'Philadelphia', 'Elkton', 'Catskill',
          'Charlottesville', 'Lynchburg', 'Newport News', 'Fredericksburg',
          'South Hill', 'Greensboro', 'Fayetteville', 'Columbia',
          'Spartanburg', 'Athens', 'Augusta', 'Chattanooga', 'Birmingham']
requests = pd.read_csv('./data/requests.csv').to_dict(orient='records')
print(f'The variable requests has a type of {type(requests)}')
print('\nLook at the first few requests:')
print(requests[0:5])

The variable requests has a type of <class 'list'>

Look at the first few requests:
[{'request_id': '2023_01_0001', 'origin_city': 'Charlottesville', 'origin_state': 'VA', 'origin_liftgate_service': 0, 'destination_city': 'Augusta', 'destintion_state': 'GA', 'destination_liftgate_service': 0, 'handling_unit': 'box', 'length': 36, 'width': 36, 'height': 24, 'weight': 495}, {'request_id': '2023_01_0002', 'origin_city': 'Columbia', 'origin_state': 'SC', 'origin_liftgate_service': 0, 'destination_city': 'Fredericksburg', 'destintion_state': 'VA', 'destination_liftgate_service': 0, 'handling_unit': 'crate', 'length': 48, 'width': 48, 'height': 72, 'weight': 3528}, {'request_id': '2023_01_0003', 'origin_city': 'Charlotte', 'origin_state': 'NC', 'origin_liftgate_service': 0, 'destination_city': 'Fayetteville', 'destintion_state': 'NC', 'destination_liftgate_service': 1, 'handling_unit': 'crate', 'length': 48, 'width': 48, 'height': 72, 'weight': 3991}, {'request_id': '2023_01_0004', 'origin_c

# 1. Find a Request

Implement the function `find_request` below that takes a request `id` as an argument and returns the entire request as a dictionary. Your function should assume that the variable `requests` is in memory. You can also assume that every request has a unique id. Be sure to read the docstring of the function.

In [3]:
def find_request(id):
    """
    A function that returns a shipping request from the list `requests`
    
    Parameters
    ----------
    id: the string id of a request to find
    
    Returns
    -------
    request: the request as a dictionary. If the request is not found return None
    """
    # initialize the return variable request to None
    request = None
    for dict in requests:
        if dict['request_id'] == id:
            request = dict
    return request

In [4]:
# This is a test cell
# If **NO** message is printed, it means that the tests passed
# One basic test to see if your function works
r = find_request('2023_01_0001')
assert_equal(type(r), dict, msg='Your function did not return a dictionary as required')
assert_equal(r['origin_city'], 'Charlottesville', msg='Your function did not return the right output')

<hr>

<hr>

# 2. Find Requests Based on Origin City

You want to find all the requests based on a specific origin city found in the request. Implement the function `get_requests_by_origin_city` below. The function should return a list of dictionaries as indicated in the docstring. **Your search must be case insensitive.** 

In [5]:
def get_requests_by_origin_city(origin_city):
    """
    A function that takes in the name of the origin city and returns 
    a list of all the requests for that particular origin city. The
    search is case insensitive.
    
    Parameters
    ----------
    origin_city: a string representing the origin city to search for
    
    Returns
    -------
    results: a list of requests, where each request is a dictionary.
             If no requests exist for the origin city, then return
             an empty list.
    """
    results = []
    for dict in requests:
        if dict.get('origin_city', '').lower() == origin_city.lower():
                results.append(dict)
    return results

In [6]:
# This is a test cell
# If **NO** message is printed, it means that the tests passed
# One basic test to see if your function works
cho = get_requests_by_origin_city('charlottesville')
assert_equal(type(cho), list, msg='Your function did not return a list as required')
assert_equal(len(cho), 10, msg='Your function did not return the correct number of requests')

<hr>

<hr>

# 3. Finding Automatically Quotable Requests

Currently, Pallet Movers manually reviews every request to generate a quote for the customer. A quote generally involves a price and estimated pickup and delivery dates based on the information that the customer provided in their request. Pallet Movers wants to streamline some of this manually intensive quoting process. Their IT department has devised a way to generate automatic quotes for a subset of requests. To use the automated quoting tool, both the origin and destination have to be part of their hub-and-spoke system. If either the origin or destination city is not in their network, the tool explodes and catches the home office on fire. Obviously, you want to avoid this situation because of your aversion to fire. 

You have been tasked to write a function that sifts through the requests, separating them into four lists. The first list should contain the requests that can be automatically quoted. The second list should contain the requests that only have an origin city **outside** of Pallet Movers network. The third list should contain the requests that only have a destination city **outside** of their network. The fourth list should contain the requests that have **both** the origin and destination cities outside of the network.

Name your function `sort_requests`. It should have three arguments in its definition: 

1. `the_requests` which is the big list of requests to sort through
2. `the_hubs` which is a list of the hub cities
3. `the_spokes` which is a list of the spoke cities

As noted above, the function should return all four lists in the order specified. You must also make sure that all of your sorting and searching is **case insensitive** to protect against a customer inadvertantly not using proper case when making the request.

In [7]:
# Create your user-defined function here
def sort_requests(the_requests, the_hubs, the_spokes):
    """
    A function that sifts through requests and separates them into four lists.
    
    Parameters:
    -----------
    the_requests: which is the big list of requests to sort through
    the_hubs: which is a list of the hub cities
    the_spokes: which is a list of the spoke cities
    
    Returns:
    ---------
    list_1: origin city and destination city inside of network.
    list_2: origin city outside, destination city inside of network.
    list_3: origin city inside, destination city outside of network.
    list_4: origin city and destination city outside of network.
    """
    
    #Convert hub and spokes to lower case for case insensitive matching
    lower_hubs = [element.lower() for element in the_hubs]
    lower_spokes = [element.lower() for element in the_spokes]
    hubs_spokes = lower_hubs + lower_spokes
    
    #Create empty lists
    list_1 = []
    list_2 = []
    list_3 = []
    list_4 = []
    
    #Covert values to lowercase
    for dict in the_requests:
        origin = dict.get('origin_city').lower()
        destination = dict.get('destination_city').lower()
        
    #Create lists based on conditions
        if origin in hubs_spokes and destination in hubs_spokes:
            list_1.append(dict)
        elif origin not in hubs_spokes and destination in hubs_spokes:
            list_2.append(dict)
        elif origin in hubs_spokes and destination not in hubs_spokes:
            list_3.append(dict)
        else:
            list_4.append(dict)
    return list_1, list_2, list_3, list_4

sort_requests(requests, hubs, spokes)
    
    

([{'request_id': '2023_01_0001',
   'origin_city': 'Charlottesville',
   'origin_state': 'VA',
   'origin_liftgate_service': 0,
   'destination_city': 'Augusta',
   'destintion_state': 'GA',
   'destination_liftgate_service': 0,
   'handling_unit': 'box',
   'length': 36,
   'width': 36,
   'height': 24,
   'weight': 495},
  {'request_id': '2023_01_0002',
   'origin_city': 'Columbia',
   'origin_state': 'SC',
   'origin_liftgate_service': 0,
   'destination_city': 'Fredericksburg',
   'destintion_state': 'VA',
   'destination_liftgate_service': 0,
   'handling_unit': 'crate',
   'length': 48,
   'width': 48,
   'height': 72,
   'weight': 3528},
  {'request_id': '2023_01_0003',
   'origin_city': 'Charlotte',
   'origin_state': 'NC',
   'origin_liftgate_service': 0,
   'destination_city': 'Fayetteville',
   'destintion_state': 'NC',
   'destination_liftgate_service': 1,
   'handling_unit': 'crate',
   'length': 48,
   'width': 48,
   'height': 72,
   'weight': 3991},
  {'request_id': '20

<hr>

<hr>

<hr>

<hr>

<hr>

**&copy; 2022 - Present: Matthew D. Dean, Ph.D.   
Clinical Associate Professor of Business Analytics at William \& Mary.**